# Surrogates, uncertainty, and Bayesian optimization

<div style="background:linear-gradient(135deg,#eef3fb 0%,#f8fafc 100%);border:1px solid #d6e0ee;border-radius:12px;padding:18px 26px;color:#1b1b1b">
  <div style="font-size:12px;letter-spacing:.12em;text-transform:uppercase;color:#1f5fa8;font-weight:700">Hands-on workshop · ~60 min</div>
  <div style="font-size:16px;color:#3a3a3a;max-width:820px;margin-top:6px">From a CFD DOE table over Mach × angle of attack to <i>“which flight condition do we simulate next?”</i></div>
</div>

**You will learn to** build a surrogate modeling pipeline (`scaler → reducer → estimator`); read μ and σ as *maps* and spot a data gap from σ alone; split uncertainty into the part more CFD runs can fix and the part they can't; turn (μ, σ) into a decision.

Every figure is interactive (hover, zoom, sliders, dropdowns). Every code cell is editable and runs live — green boxes are exercises with a collapsed solution.

In [1]:
import uuid
import warnings
from dataclasses import dataclass
from enum import IntEnum, ReprEnum, StrEnum

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from numpy.typing import NDArray
from plotly.offline import get_plotlyjs
from plotly.subplots import make_subplots
from scipy.interpolate import RegularGridInterpolator
from scipy.linalg import cho_solve
from scipy.stats import norm
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

type FloatArray = NDArray[np.float64]
type Trace = go.Heatmap | go.Scatter


class Aero(float, ReprEnum):
    """Design box, atmosphere and the buffet onset.

    Attributes:
        MACH_MIN: Lower Mach bound.
        MACH_MAX: Upper Mach bound.
        AOA_MIN: Lower angle of attack, degrees.
        AOA_MAX: Upper angle of attack, degrees.
        SPEED_OF_SOUND: m/s.
        DENSITY: kg/m³.
        VISCOSITY: Pa·s.
        BUFFET_MACH: Mach where shock buffet starts.
    """

    MACH_MIN = 0.2
    MACH_MAX = 0.95
    AOA_MIN = -2.0
    AOA_MAX = 16.0
    SPEED_OF_SOUND = 340.0
    DENSITY = 1.225
    VISCOSITY = 1.8e-5
    BUFFET_MACH = 0.84


class Files(StrEnum):
    """Workshop data files and where to find them.

    Attributes:
        RUNS: The CFD run log (the DOE).
        REFERENCE: Dense reference sweep used only for grading.
        REMOTE: Base URL used when the files are not next to the notebook.
    """

    RUNS = "cfd_runs.csv"
    REFERENCE = "cfd_reference.csv"
    REMOTE = "https://raw.githubusercontent.com/horizonsn59-code/workshop-nb/main/data/"


class Sizes(IntEnum):
    """Sample sizes and seeds.

    Attributes:
        GRID: Points per axis of the reference sweep.
        LHS: Runs in the initial Latin hypercube (the first rows of the DOE).
        DEMO: DOE rows used for the single-surrogate figures and exercises.
        BO_INIT: BO initial design size (the Latin hypercube).
        BO_ITER: BO acquisition steps.
        BO_SEED: Observation-noise seed of the BO run.
        HEIGHT: Figure height in pixels.
    """

    GRID = 60
    LHS = 24
    DEMO = 96
    BO_INIT = 24
    BO_ITER = 16
    BO_SEED = 5
    HEIGHT = 480


class Acquisition(StrEnum):
    """Acquisition rules for run_bo.

    Attributes:
        EI: Expected improvement.
        PI: Probability of improvement.
        UCB: Upper confidence bound.
    """

    EI = "EI"
    PI = "PI"
    UCB = "UCB"


warnings.filterwarnings("ignore", category=ConvergenceWarning)
TRAIN_SIZES = (24, 48, 96, 192)
FEATURES = ["mach", "aoa", "v_inf", "q_inf", "reynolds"]
PLOTLY_JS_EMITTED = [False]


def load(name: Files) -> pd.DataFrame:
    """Read a workshop CSV from next to the notebook, from the repository's data folder, or from the remote copy.

    Args:
        name: File to load.

    Returns:
        The table.
    """
    for folder in ("data", "../data", "."):
        try:
            return pd.read_csv(f"{folder}/{name}")
        except FileNotFoundError:
            continue
    return pd.read_csv(Files.REMOTE + name)


def show(fig: go.Figure, title: str, subtitle: str = "") -> None:
    """Light theme, title, and render as HTML that draws once its element is in the page (Jupyter, Thebe, nbconvert, browser build).

    Args:
        fig: Figure to render.
        title: Headline.
        subtitle: Muted line under the headline.
    """
    sub = f"<br><sup><span style='color:#666'>{subtitle}</span></sup>" if subtitle else ""
    fig.update_layout(template="plotly_white", paper_bgcolor="white", plot_bgcolor="white", font=dict(size=13),
                      title=dict(text=f"<b>{title}</b>{sub}", x=0.0, xanchor="left", font=dict(size=17)))
    uid = uuid.uuid4().hex
    draw = (f'<script>(function(){{var fig={fig.to_json()};var tries=0;(function draw(){{var el=document.getElementById("{uid}");'
            f'if(!el||!window.Plotly){{if(tries++<400)setTimeout(draw,25);return;}}'
            f'Plotly.newPlot(el,fig.data,fig.layout,{{displaylogo:false,responsive:true}}).then(function(){{if(fig.frames)Plotly.addFrames(el,fig.frames);}});}})();}})();</script>')
    plotly_js = "" if PLOTLY_JS_EMITTED[0] else f"<script>{get_plotlyjs()}</script>"
    PLOTLY_JS_EMITTED[0] = True
    display(HTML(f'{plotly_js}<div id="{uid}"></div>{draw}'))


def animate(fig: go.Figure, frames: list[list[Trace]], labels: list[str], prefix: str = "", dropdown: bool = False) -> None:
    """Expose per-step trace lists through a slider or a dropdown.

    Args:
        fig: Figure whose base traces equal frames[0].
        frames: One trace list per step.
        labels: One label per step.
        prefix: Text before the active label (slider only).
        dropdown: Use a dropdown instead of a slider.
    """
    fig.frames = [go.Frame(data=data, name=label) for data, label in zip(frames, labels)]
    opts = dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))
    entries = [dict(method="animate", args=[[label], opts], label=label) for label in labels]
    if dropdown:
        fig.update_layout(updatemenus=[dict(buttons=entries, x=0, xanchor="left", y=1.2, yanchor="top")], margin=dict(t=120))
    else:
        fig.update_layout(sliders=[dict(steps=entries, currentvalue=dict(prefix=prefix), pad=dict(t=110), x=0.02, len=0.96)])

## 1. The data: a minimal Latin hypercube, extended by Bayesian-optimization infill

Two files, both plain CSV:

* **`cfd_runs.csv`** — 192 CFD runs of one wing, in the order they were made: a 24-run **Latin hypercube** over Mach and angle of attack (`phase = lhs`), then 168 **infill** runs placed one at a time by a Bayesian optimizer looking for the best cruise condition (`phase = infill`). Each row is what the run log records: the two conditions that were set, $M$ and $\alpha$, three derived flow quantities ($V_\infty$, $q_\infty$, $Re$), and the result, the lift-to-drag ratio $L/D$.
* **`cfd_reference.csv`** — a dense 60 × 60 sweep of the same wing: the noise-free $L/D$, the solver's noise level, and one noisy run at every point. In practice this does not exist — it is 3,600 runs — and it is used here only to grade the surrogate, never to train it.

Three things to notice before fitting anything.

In [2]:
RUNS = load(Files.RUNS)
REF = load(Files.REFERENCE)
MACH_AXIS, AOA_AXIS = np.unique(REF["mach"]), np.unique(REF["aoa"])
MACH_GRID, AOA_GRID = REF["mach"].to_numpy(), REF["aoa"].to_numpy()
X_GRID, Y_TRUE, SIGMA_TRUE = REF[FEATURES].to_numpy(), REF["ld_true"].to_numpy(), REF["noise_sigma"].to_numpy()
MACH_POOL, AOA_POOL, X_POOL, Y_POOL = RUNS["mach"].to_numpy(), RUNS["aoa"].to_numpy(), RUNS[FEATURES].to_numpy(), RUNS["lift_to_drag"].to_numpy()
I_OPT = int(np.argmax(Y_TRUE))


def doe_table(mach: FloatArray, aoa: FloatArray) -> FloatArray:
    """Rebuild the five run-log columns for new conditions: M, α, V∞, q∞, Re.

    Args:
        mach: Mach numbers.
        aoa: Angles of attack in degrees.

    Returns:
        Feature matrix (n, 5) in raw units.
    """
    velocity = mach * Aero.SPEED_OF_SOUND
    return np.column_stack([mach, aoa, velocity, 0.5 * Aero.DENSITY * velocity ** 2, Aero.DENSITY * velocity / Aero.VISCOSITY])


def nearest_run(mach: FloatArray, aoa: FloatArray, n: int = Sizes.DEMO) -> FloatArray:
    """Distance from each condition to the closest of the first n runs, in scaled design-box units.

    Args:
        mach: Mach numbers.
        aoa: Angles of attack in degrees.
        n: DOE rows considered.

    Returns:
        Distance per condition (1 = the full width of the box).
    """
    dm = (mach[:, None] - MACH_POOL[None, :n]) / (Aero.MACH_MAX - Aero.MACH_MIN)
    da = (aoa[:, None] - AOA_POOL[None, :n]) / (Aero.AOA_MAX - Aero.AOA_MIN)
    return np.sqrt(dm ** 2 + da ** 2).min(axis=1)


_TRUE = RegularGridInterpolator((AOA_AXIS, MACH_AXIS), Y_TRUE.reshape(Sizes.GRID, Sizes.GRID))


def true_ld(mach: FloatArray, aoa: FloatArray) -> FloatArray:
    """Noise-free L/D at arbitrary conditions, interpolated from the reference sweep.

    Args:
        mach: Mach numbers.
        aoa: Angles of attack in degrees.

    Returns:
        Lift-to-drag ratio.
    """
    return _TRUE(np.column_stack([aoa, mach]))


def cfd_run(index: NDArray[np.int64], rng: np.random.Generator) -> FloatArray:
    """Stand-in for launching a CFD job at reference-grid points: the true value plus the solver's noise.

    Args:
        index: Reference-grid indices to run.
        rng: Noise generator.

    Returns:
        Noisy L/D per run.
    """
    return Y_TRUE[index] + rng.normal(size=np.size(index)) * SIGMA_TRUE[index]


print(f"{len(RUNS)} runs · {RUNS['mach'].min():.2f} ≤ M ≤ {RUNS['mach'].max():.2f} · {RUNS['aoa'].min():.0f}° ≤ α ≤ {RUNS['aoa'].max():.0f}°")
print(RUNS["phase"].value_counts().to_dict(), "  ·  best L/D in the log:", f"{Y_POOL.max():.1f}", f"(run {int(RUNS.loc[Y_POOL.argmax(), 'run_id']):d}, {RUNS.loc[Y_POOL.argmax(), 'phase']})")
print(f"true optimum (reference sweep): L/D = {Y_TRUE[I_OPT]:.2f} at Mach {MACH_GRID[I_OPT]:.2f}, AoA {AOA_GRID[I_OPT]:.1f}°"
      f" — nearest Latin-hypercube run is {nearest_run(MACH_GRID[[I_OPT]], AOA_GRID[[I_OPT]], Sizes.LHS)[0]:.2f} box-widths away")
RUNS.head()

192 runs · 0.20 ≤ M ≤ 0.95 · -2° ≤ α ≤ 16°
{'infill': 168, 'lhs': 24}   ·  best L/D in the log: 20.7 (run 68, infill)
true optimum (reference sweep): L/D = 20.50 at Mach 0.76, AoA 1.7° — nearest Latin-hypercube run is 0.07 box-widths away


,run_id,phase,mach,aoa,v_inf,q_inf,reynolds,lift_to_drag
0,1,lhs,0.541073,11.322400,183.965,20728.90,12519800.0,15.57570
1,2,lhs,0.331210,-0.936622,112.611,7767.32,7663840.0,5.12403
2,3,lhs,0.853308,13.425200,290.125,51555.60,19744600.0,2.88189
3,4,lhs,0.747530,14.380200,254.160,39565.90,17297000.0,5.35213
4,5,lhs,0.364544,15.164700,123.945,9409.47,8435150.0,8.83114


**1. Five columns, not two.** $V_\infty$, $q_\infty$ and $Re$ are all functions of Mach — the correlation matrix says so — with magnitudes from $10^0$ to $10^7$. A model that treats them as five independent inputs is wasting four of them, and one that is not scaled first will be dominated by the Reynolds column.

**2. The response is not flat.** $L/D$ runs from below zero (negative lift) to about 20; the long upper tail is the small cruise region every design study is looking for.

**3. Coverage is very uneven.** The Latin hypercube puts one run in each of 24 Mach bands and 24 AoA bands — thin everywhere. The infill then piles onto the high-$L/D$ ridge around $M \approx 0.7$, $\alpha \approx 2°$, which is what an optimizer is for, and leaves stall and buffet almost untouched. Those are the data gaps a surrogate has to be honest about. The noise is a map, not a number: clean attached flow converges tightly, stall and shock buffet ($M \gtrsim 0.84$) do not.

In [3]:
print(RUNS[FEATURES + ["lift_to_drag"]].describe().loc[["mean", "std", "min", "max"]].round(2))
print("\ncorrelation with Mach:", RUNS[FEATURES].corr().loc["mach"].round(3).to_dict())

fig = make_subplots(rows=1, cols=2, subplot_titles=("the 192 runs over the design box", "distribution of L/D in the run log"), column_widths=[0.55, 0.45], horizontal_spacing=0.12)
LHS_ROWS = RUNS["phase"] == "lhs"
fig.add_trace(go.Scatter(x=MACH_POOL[~LHS_ROWS], y=AOA_POOL[~LHS_ROWS], mode="markers", name="168 infill runs (BO)", marker=dict(color="#1f5fa8", size=6, opacity=0.6),
                         hovertemplate="run %{customdata}<br>Mach %{x:.2f}<br>AoA %{y:.1f}°<extra></extra>", customdata=RUNS.loc[~LHS_ROWS, "run_id"]), row=1, col=1)
fig.add_trace(go.Scatter(x=MACH_POOL[LHS_ROWS], y=AOA_POOL[LHS_ROWS], mode="markers", name="24-run Latin hypercube", marker=dict(color="#d1495b", size=9, symbol="diamond", line=dict(color="white", width=1)),
                         hovertemplate="run %{customdata}<br>Mach %{x:.2f}<br>AoA %{y:.1f}°<extra></extra>", customdata=RUNS.loc[LHS_ROWS, "run_id"]), row=1, col=1)
fig.add_trace(go.Histogram(x=Y_POOL, nbinsx=30, marker_color="#1f5fa8", showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="Mach", row=1, col=1); fig.update_yaxes(title_text="angle of attack [deg]", row=1, col=1)
fig.update_xaxes(title_text="L/D", row=1, col=2); fig.update_yaxes(title_text="runs", row=1, col=2)
fig.update_layout(height=Sizes.HEIGHT - 60, legend=dict(orientation="h", y=-0.2, x=0))
show(fig, "What the run log contains", "diamonds: the space-filling start · dots: where the optimizer then sent the runs")

      mach    aoa   v_inf     q_inf     reynolds  lift_to_drag
mean  0.68   3.11  230.78  33786.73  15705772.60         18.23
std   0.13   2.98   43.74  10624.39   2976732.54          3.92
min   0.20  -2.00   68.00   2832.20   4627780.00         -0.06
max   0.95  16.00  323.00  63901.50  21981900.00         20.71

correlation with Mach: {'mach': 1.0, 'aoa': -0.254, 'v_inf': 1.0, 'q_inf': 0.981, 'reynolds': 1.0}


In [4]:
def fields_figure(titles: tuple[str, ...]) -> go.Figure:
    """Empty 1×k subplot figure with Mach / AoA axes.

    Args:
        titles: One subplot title per column.

    Returns:
        Figure ready for field and points traces.
    """
    fig = make_subplots(rows=1, cols=len(titles), subplot_titles=titles, horizontal_spacing=0.36 / len(titles))
    fig.update_xaxes(title_text="Mach")
    fig.update_yaxes(title_text="angle of attack [deg]", col=1)
    fig.update_layout(height=Sizes.HEIGHT, legend=dict(orientation="h", y=-0.15, x=0))
    return fig


def field(fig: go.Figure, z: FloatArray, col: int, colorscale: str, zrange: tuple[float, float] | None = None, name: str = "") -> go.Heatmap:
    """Design-space heatmap for subplot column col, colourbar parked right of that panel.

    Args:
        fig: Figure the trace will live in.
        z: Values on the flattened grid.
        col: 1-based subplot column.
        colorscale: Plotly colourscale.
        zrange: Fixed colour range.
        name: Colourbar title.

    Returns:
        Heatmap trace bound to that column's axes.
    """
    suffix = "" if col == 1 else str(col)
    zmin, zmax = zrange if zrange else (None, None)
    return go.Heatmap(x=MACH_AXIS, y=AOA_AXIS, z=z.reshape(Sizes.GRID, Sizes.GRID), colorscale=colorscale, zmin=zmin, zmax=zmax, zsmooth="best",
                      xaxis=f"x{suffix}", yaxis=f"y{suffix}", colorbar=dict(title=name, len=0.8, thickness=14, x=fig.layout[f"xaxis{suffix}"].domain[1] + 0.012),
                      hovertemplate="Mach %{x:.2f}<br>AoA %{y:.1f}°<br>%{z:.2f}<extra></extra>")


def points(mach: FloatArray, aoa: FloatArray, col: int, name: str = "CFD runs", color: str = "#d1495b", size: int = 7, symbol: str = "circle") -> go.Scatter:
    """Markers for subplot column col.

    Args:
        mach: Mach numbers.
        aoa: Angles of attack.
        col: 1-based subplot column.
        name: Legend name.
        color: Marker colour.
        size: Marker size.
        symbol: Marker symbol.

    Returns:
        Scatter trace bound to that column's axes.
    """
    suffix = "" if col == 1 else str(col)
    return go.Scatter(x=mach, y=aoa, mode="markers", name=name, xaxis=f"x{suffix}", yaxis=f"y{suffix}", showlegend=col == 1,
                      marker=dict(color=color, size=size, symbol=symbol, line=dict(color="white", width=1)), hovertemplate="Mach %{x:.2f}<br>AoA %{y:.1f}°<extra></extra>")


fig = fields_figure(("true L/D", "true noise σ"))
fig.add_traces([field(fig, Y_TRUE, 1, "Viridis", name="L/D"), points(MACH_POOL[:Sizes.LHS], AOA_POOL[:Sizes.LHS], 1, name="Latin hypercube", symbol="diamond", size=9),
                field(fig, SIGMA_TRUE, 2, "YlOrRd", name="σ"), points(MACH_POOL[:Sizes.LHS], AOA_POOL[:Sizes.LHS], 2, symbol="diamond", size=9)])
show(fig, "What the reference sweep shows", "left: the field the surrogate has to learn, with the 24 starting runs · right: how noisy the solver is — neither map exists in a real study")

In [5]:
fig = fields_figure(("the 24-run Latin hypercube, as measured — no model, no interpolation",))
fig.add_trace(go.Scatter(x=MACH_POOL[:Sizes.LHS], y=AOA_POOL[:Sizes.LHS], mode="markers+text", text=[f"{v:.1f}" for v in Y_POOL[:Sizes.LHS]], textposition="top center", textfont=dict(size=10, color="#444"),
                         marker=dict(color=Y_POOL[:Sizes.LHS], colorscale="Viridis", size=13, symbol="diamond", colorbar=dict(title="L/D", len=0.8, thickness=14), line=dict(color="white", width=1)),
                         hovertemplate="run %{customdata}<br>Mach %{x:.2f}<br>AoA %{y:.1f}°<extra></extra>", customdata=RUNS["run_id"][:Sizes.LHS], showlegend=False))
fig.update_layout(height=Sizes.HEIGHT, xaxis_range=[Aero.MACH_MIN - 0.03, Aero.MACH_MAX + 0.03], yaxis_range=[Aero.AOA_MIN - 1, Aero.AOA_MAX + 1])
show(fig, "Where the campaign started", "one run per Mach band and per AoA band; the number is the measured L/D — everything between the points is unknown")

## 2. The pipeline: scaler → reducer → estimator

One `Pipeline`, three named steps, fitted together:



Scale first: on the raw table the $Re$ column ($10^7$) *is* the variance, so PCA would keep Reynolds number and drop $\alpha$.

The scaler and the reducer learn from data too, so they must be fitted on training rows only. A `Pipeline` guarantees that: `fit` refits every step on the rows it is given, `predict` only transforms — which is what keeps cross-validation honest.



In [6]:
def build_pipeline(n_components: int = 2, seed: int = 0) -> Pipeline:
    """Scaler → PCA → Gaussian process.

    Args:
        n_components: Principal components kept.
        seed: Seed for the GP optimizer restarts.

    Returns:
        Unfitted pipeline.
    """
    kernel = ConstantKernel(1.0, (1e-2, 1e3)) * RBF([0.5] * n_components, (1e-2, 5.0)) + WhiteKernel(0.1, (1e-5, 1e1))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=3, random_state=seed)
    return Pipeline([("scaler", StandardScaler()), ("reducer", PCA(n_components=n_components)), ("estimator", gp)])


def predict_parts(pipe: Pipeline, X: FloatArray) -> tuple[FloatArray, FloatArray, FloatArray]:
    """Mean, epistemic std and total std; total² = epistemic² + fitted noise variance.

    Args:
        pipe: Fitted pipeline.
        X: Feature matrix.

    Returns:
        Tuple of (mean, epistemic std, total std).
    """
    mean, std_total = pipe.predict(X, return_std=True)
    gp = pipe[-1]
    noise_var = float(gp.kernel_.k2.noise_level) * float(gp._y_train_std ** 2)
    return mean, np.sqrt(np.clip(std_total ** 2 - noise_var, 1e-12, None)), std_total


labels = [f"PC{i + 1}" for i in range(5)]
fig = go.Figure([go.Bar(x=labels, y=PCA().fit(X_POOL).explained_variance_ratio_, name="PCA on the raw table", marker_color="#b8c4d0"),
                 go.Bar(x=labels, y=PCA().fit(StandardScaler().fit_transform(X_POOL)).explained_variance_ratio_, name="PCA after StandardScaler", marker_color="#1f5fa8")])
fig.update_layout(height=360, yaxis_title="explained variance ratio", legend=dict(orientation="h", y=1.02, x=1, xanchor="right"))
show(fig, "Explained variance per component", "raw: PC1 is the Reynolds column · scaled: PC1 = speed, PC2 = angle of attack")

pipe_demo = build_pipeline().fit(X_POOL[:Sizes.DEMO], Y_POOL[:Sizes.DEMO])
latent = pipe_demo[:-1].transform(X_POOL[:Sizes.DEMO])
print("|corr(PC1, Mach)| =", round(abs(np.corrcoef(latent[:, 0], MACH_POOL[:Sizes.DEMO])[0, 1]), 3), "  |corr(PC2, AoA)| =", round(abs(np.corrcoef(latent[:, 1], AOA_POOL[:Sizes.DEMO])[0, 1]), 3))

|corr(PC1, Mach)| = 0.998   |corr(PC2, AoA)| = 0.959


<div style="border-left:5px solid #2a9d8f;background:#eef8f6;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>🔧 Your turn 1 — walk a sample through the pipeline (5 min)</b><br><ol style='margin:6px 0 0 18px'><li>Which column has the largest <code>scale_</code>? Push three rows through <code>pipe_demo[:1]</code>, then <code>pipe_demo[:2]</code>.</li><li>Find the <code>get_params()</code> key for the number of PCA components.</li><li>Refit with <code>n_components=1</code> and compare the RMSE. What got thrown away?</li></ol></div>

In [7]:
print("stages:", list(pipe_demo.named_steps))
print("scaler scale_:", pipe_demo.named_steps["scaler"].scale_.round(2))
print("after scaler :", pipe_demo[:1].transform(X_POOL[:3]).round(2))
print("after reducer:", pipe_demo[:2].transform(X_POOL[:3]).round(2))
print("reducer params:", [k for k in pipe_demo.get_params() if k.startswith("reducer__")])
rmse = lambda pipe: np.sqrt(np.mean((pipe.predict(X_GRID) - Y_TRUE) ** 2))
print(f"RMSE vs truth, 2 components: {rmse(pipe_demo):.2f}")
# TODO: refit with reducer__n_components=1 and print its RMSE

stages: ['scaler', 'reducer', 'estimator']
scaler scale_: [1.60000000e-01 3.93000000e+00 5.53600000e+01 1.29440500e+04
 3.76787457e+06]
after scaler : [[-0.83  1.97 -0.83 -1.05 -0.83]
 [-2.12 -1.14 -2.12 -2.05 -2.12]
 [ 1.09  2.51  1.09  1.33  1.09]]
after reducer: [[-2.03  1.69]
 [-4.   -1.72]
 [ 1.92  2.81]]
reducer params: ['reducer__copy', 'reducer__iterated_power', 'reducer__n_components', 'reducer__n_oversamples', 'reducer__power_iteration_normalizer', 'reducer__random_state', 'reducer__svd_solver', 'reducer__tol', 'reducer__whiten']
RMSE vs truth, 2 components: 1.88


<details style="margin:10px 0"><summary style="cursor:pointer;color:#1f5fa8;font-weight:600">Solution</summary><pre style="background:#f5f7f9;border:1px solid #dde3ea;border-radius:6px;padding:12px;font-size:12.5px"><code>pipe_1 = build_pipeline(n_components=1).fit(X_POOL[:Sizes.DEMO], Y_POOL[:Sizes.DEMO])
print(f&quot;RMSE vs truth, 1 component: {rmse(pipe_1):.2f}&quot;)   # ≈ 5.7 vs 1.9: AoA is gone, and L/D is mostly an AoA story</code></pre><p style='margin:8px 0 0 0'><code>set_params(step__param=…)</code> is how a hyperparameter search tunes a pipeline — same skeleton, different parameters.</p></details>

## 3. What a surrogate returns: `predict(X)` vs `predict(X, return_std=True)`

A fitted pipeline is called like any scikit-learn model. The one difference that matters for everything that follows is the `return_std` flag.
<div style="border-left:5px solid #2a9d8f;background:#eef8f6;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>🔧 Your turn 2 — call predict both ways (5 min)</b><br><code>X_QUERY</code> holds five flight conditions: two inside the infill cluster, one in sparsely covered subsonic flow, one deep in stall, one in buffet.<ol style='margin:6px 0 0 18px'><li>Run the cell: <code>predict(X_QUERY)</code> gives one array — the mean μ.</li><li>Change it to <code>predict(X_QUERY, return_std=True)</code>. You now get a <i>tuple</i> (μ, σ); unpack it and print both.</li><li>Which row has the largest σ, and why? Which has the smallest? Compare μ with the true L/D printed alongside.</li><li>Try <code>return_cov=True</code> instead — what shape comes back, and what are its diagonal entries?</li></ol></div>

In [8]:
QUERY_MACH = np.array([0.72, 0.65, 0.35, 0.90, 0.93])
QUERY_AOA = np.array([2.0, 3.0, 12.0, 13.0, 3.0])
X_QUERY = doe_table(QUERY_MACH, QUERY_AOA)
np.set_printoptions(precision=2, suppress=True)

print("X_QUERY (M, α, V∞, q∞, Re):\n", X_QUERY)
print("true L/D       :", true_ld(QUERY_MACH, QUERY_AOA))
print("predict(X)     :", pipe_demo.predict(X_QUERY))
# TODO: mean, std = pipe_demo.predict(X_QUERY, return_std=True)

X_QUERY (M, α, V∞, q∞, Re):
 [[       0.72        2.        244.8     36705.31 16660000.  ]
 [       0.65        3.        221.      29915.11 15040277.78]
 [       0.35       12.        119.       8673.61  8098611.11]
 [       0.9        13.        306.      57352.05 20825000.  ]
 [       0.93        3.        316.2     61239.24 21519166.67]]
true L/D       : [20.21 19.42 13.98  1.27  4.94]
predict(X)     : [20.38 19.31 13.87  4.35  2.9 ]


<details style="margin:10px 0"><summary style="cursor:pointer;color:#1f5fa8;font-weight:600">Solution</summary><pre style="background:#f5f7f9;border:1px solid #dde3ea;border-radius:6px;padding:12px;font-size:12.5px"><code>mean, std = pipe_demo.predict(X_QUERY, return_std=True)
print(&quot;mean μ:&quot;, mean)
print(&quot;std  σ:&quot;, std)
print(&quot;distance to nearest run:&quot;, nearest_run(QUERY_MACH, QUERY_AOA).round(2))
cov = pipe_demo.predict(X_QUERY, return_cov=True)[1]
print(&quot;cov shape:&quot;, cov.shape, &quot;| sqrt(diag) == σ:&quot;, np.allclose(np.sqrt(np.diag(cov)), std))</code></pre><p style='margin:8px 0 0 0'>Rows 1–2 sit inside the infill cluster: σ is at the noise floor. Rows 3–5 are far from any run — high angle of attack, stall, buffet — and σ grows with that distance. Compare the <code>distance to nearest run</code> line with σ: the surrogate is saying where the data runs out. <code>return_std</code> is the whole reason a surrogate is more than a curve fit.</p></details>

### Mean and σ as maps

Call `predict(X, return_std=True)` on a whole grid and the two vectors become two *fields*: $\mu(x)$, the best guess, and $\sigma(x)$, how far the truth could plausibly be from it. For a GP,

$$
\sigma^2(x_*) = \underbrace{k(x_*, x_*) - k_*^\top (K + \sigma_n^2 I)^{-1} k_*}_{\text{shrinks as data arrives}} + \underbrace{\sigma_n^2}_{\text{never shrinks}} .
$$

Drag the slider from the 24-run Latin hypercube through the infill: μ sharpens along the ridge the optimizer chased, and σ collapses there — but stays bright in stall and buffet, however many runs land on the ridge. Between the 24 starting points the surrogate is guessing, and it says so.

In [9]:
FITS = {n: build_pipeline().fit(X_POOL[:n], Y_POOL[:n]) for n in TRAIN_SIZES}
PARTS = {n: predict_parts(FITS[n], X_GRID) for n in TRAIN_SIZES}
STD_RANGE = (0.0, float(SIGMA_TRUE.max()))
LD_RANGE = (float(Y_TRUE.min()), float(Y_TRUE.max()))

fig = fields_figure(("surrogate mean μ", "surrogate std σ"))
def runs_overlay(n: int, col: int) -> list[go.Scatter]:
    """Latin-hypercube runs as diamonds and infill runs as dots, for the first n rows.

    Args:
        n: DOE rows shown.
        col: 1-based subplot column.

    Returns:
        Two scatter traces.
    """
    k = min(n, Sizes.LHS)
    return [points(MACH_POOL[:k], AOA_POOL[:k], col, name="Latin hypercube", symbol="diamond", size=9),
            points(MACH_POOL[k:n], AOA_POOL[k:n], col, name="BO infill", color="#1f5fa8", size=6)]


STAGE_LABELS = [f"{n} · LHS" if n == Sizes.LHS else f"{n} · +{n - Sizes.LHS} infill" for n in TRAIN_SIZES]
frames = [[field(fig, PARTS[n][0], 1, "Viridis", LD_RANGE, "L/D"), *runs_overlay(n, 1),
           field(fig, PARTS[n][2], 2, "YlOrRd", STD_RANGE, "σ"), *runs_overlay(n, 2)] for n in TRAIN_SIZES]
fig.add_traces(frames[0])
animate(fig, frames, STAGE_LABELS, prefix="CFD runs: ")
fig.update_layout(height=Sizes.HEIGHT + 110)
show(fig, "Mean and σ fields: the Latin hypercube first, then the infill", "σ is the map of where the model has never been — watch it collapse along the ridge and stay bright elsewhere")

Same surrogate read the aero way — $L/D$ versus $\alpha$ at fixed Mach, with the runs within ±0.05 Mach. Inner band: uncertainty from *missing data*; outer band: plus the fitted noise. Try $M = 0.75$ (dense infill at low α, nothing above 6°) and $M = 0.90$ (buffet).

In [10]:
def slice_frame(mach: float) -> list[Trace]:
    """L/D vs AoA at fixed Mach for the demo surrogate: truth, bands, mean, nearby runs.

    Args:
        mach: Mach number of the slice.

    Returns:
        Trace list for one dropdown entry.
    """
    mean, std_epi, std_total = predict_parts(FITS[Sizes.DEMO], doe_table(np.full_like(AOA_AXIS, mach), AOA_AXIS))
    near = np.abs(MACH_POOL[:Sizes.DEMO] - mach) < 0.05
    closed = np.concatenate([AOA_AXIS, AOA_AXIS[::-1]])
    band = lambda s: np.concatenate([mean + 2 * s, (mean - 2 * s)[::-1]])
    return [go.Scatter(x=closed, y=band(std_total), fill="toself", fillcolor="rgba(237,139,0,0.25)", line_width=0, hoverinfo="skip", name="μ ± 2σ total"),
            go.Scatter(x=closed, y=band(std_epi), fill="toself", fillcolor="rgba(31,95,168,0.35)", line_width=0, hoverinfo="skip", name="μ ± 2σ epistemic"),
            go.Scatter(x=AOA_AXIS, y=true_ld(np.full_like(AOA_AXIS, mach), AOA_AXIS), line=dict(color="#4a4a4a", dash="dash"), name="true L/D"),
            go.Scatter(x=AOA_AXIS, y=mean, line=dict(color="#1f5fa8", width=2.5), name="surrogate mean μ"),
            go.Scatter(x=AOA_POOL[:Sizes.DEMO][near], y=Y_POOL[:Sizes.DEMO][near], mode="markers", marker=dict(color="#d1495b", size=7), name="runs within ±0.05 Mach")]


MACH_SLICES = (0.3, 0.6, 0.75, 0.9)
fig = go.Figure(slice_frame(MACH_SLICES[0]))
animate(fig, [slice_frame(m) for m in MACH_SLICES], [f"Mach {m:.2f}" for m in MACH_SLICES], dropdown=True)
fig.update_layout(height=Sizes.HEIGHT, xaxis_title="angle of attack [deg]", yaxis_title="L/D", hovermode="x unified", legend=dict(orientation="h", y=-0.2))
show(fig, "L/D versus angle of attack at fixed Mach")

<div style="border-left:5px solid #2a9d8f;background:#eef8f6;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>🔧 Your turn 3 — are the bands honest? (5 min)</b><br><code>coverage</code> uses the one noisy run at every reference point and measures how often it falls inside μ ± 2σ (calibrated ≈ 0.95).<ol style='margin:6px 0 0 18px'><li>Whole design space: anywhere near 0.95?</li><li>Now with the <code>CLEAN</code> and <code>BUFFET</code> masks. What happens, and which assumption in <code>build_pipeline</code> is to blame?</li></ol></div>

In [11]:
X_TEST, Y_TEST = X_GRID, REF["ld_cfd"].to_numpy()
CLEAN = (AOA_GRID < 8) & (MACH_GRID < 0.8)
BUFFET = MACH_GRID >= Aero.BUFFET_MACH


def coverage(pipe: Pipeline, mask: NDArray[np.bool_] | None = None) -> float:
    """Fraction of the reference sweep's noisy runs inside μ ± 2σ.

    Args:
        pipe: Fitted pipeline.
        mask: Optional subset of the test runs.

    Returns:
        Empirical coverage.
    """
    keep = np.ones(Y_TEST.size, dtype=bool) if mask is None else mask
    mean, std = pipe.predict(X_TEST[keep], return_std=True)
    return float(np.mean(np.abs(Y_TEST[keep] - mean) <= 2 * std))


print(f"coverage over the whole design space: {coverage(FITS[Sizes.DEMO]):.3f}")
# TODO: coverage(FITS[Sizes.DEMO], CLEAN) and coverage(FITS[Sizes.DEMO], BUFFET)

coverage over the whole design space: 0.685


<details style="margin:10px 0"><summary style="cursor:pointer;color:#1f5fa8;font-weight:600">Solution</summary><pre style="background:#f5f7f9;border:1px solid #dde3ea;border-radius:6px;padding:12px;font-size:12.5px"><code>print(&quot;clean subsonic:&quot;, round(coverage(FITS[Sizes.DEMO], CLEAN), 3))   # ≈ 0.8 — bands somewhat too narrow
print(&quot;buffet:        &quot;, round(coverage(FITS[Sizes.DEMO], BUFFET), 3))  # ≈ 0.4 — bands far too narrow</code></pre><p style='margin:8px 0 0 0'>One <code>WhiteKernel</code> noise level, fitted where the runs are — on the quiet ridge, where 160 of the 192 runs sit. A homoscedastic assumption on heteroscedastic data: the bands are too narrow everywhere else and badly so in buffet, and the headline 0.69 hides where.</p></details><div style="border-left:5px solid #1f5fa8;background:#eef3fb;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>In practice</b><br>Score coverage and CRPS alongside RMSE when selecting a surrogate, and use conformal recalibration to fix band width post hoc for any model family.</div>

## 4. Uncertainty decomposition



$$
\underbrace{\mathrm{Var}[y \mid x]}_{\text{total}} = \underbrace{\mathbb{E}_\theta[\mathrm{Var}[y \mid x, \theta]]}_{\text{aleatoric}} + \underbrace{\mathrm{Var}_\theta[\mathbb{E}[y \mid x, \theta]]}_{\text{epistemic}}
$$

**Disentangling, two-stage:** a GP on $y$ gives $\mu$ and the epistemic variance; a second GP on the log squared **leave-one-out** residuals (closed form for a GP — in-sample residuals won't do, a GP's mean passes almost through its own points) gives a smooth noise map. Other routes to the same split: heteroscedastic GPs, BNN ensembles, conformal prediction.

Drag the slider: epistemic (left) collapses where the infill lands and nowhere else; aleatoric (middle) converges to the true noise map (right) only where there are runs to learn it from. With 24–48 runs the split is mostly noise; it becomes trustworthy around a hundred.

In [12]:
def noise_map(pipe: Pipeline, n: int) -> FloatArray:
    """Aleatoric std map from a second GP on log squared leave-one-out residuals, net of the epistemic part.

    Args:
        pipe: Pipeline fitted on the first n pool rows.
        n: Pool rows used.

    Returns:
        Aleatoric std on the grid.
    """
    gp = pipe[-1]
    precision = np.diag(cho_solve((gp.L_, True), np.eye(n)))
    scale = float(gp._y_train_std)
    residual = gp.alpha_.ravel() / precision * scale
    var_epi = (1.0 / precision - float(gp.kernel_.k2.noise_level)) * scale ** 2
    excess = np.maximum(residual ** 2 - var_epi, 0.25 * residual ** 2)
    return np.sqrt(np.exp(build_pipeline().fit(X_POOL[:n], np.log(excess + 1e-3)).predict(X_GRID) + 1.27))


fig = fields_figure(("epistemic std", "aleatoric std (learned)", "true noise σ"))
frames = [[field(fig, PARTS[n][1], 1, "YlOrRd", STD_RANGE, "σ"), *runs_overlay(n, 1),
           field(fig, noise_map(FITS[n], n), 2, "YlOrRd", STD_RANGE, "σ"), *runs_overlay(n, 2),
           field(fig, SIGMA_TRUE, 3, "YlOrRd", STD_RANGE, "σ"), *runs_overlay(n, 3)] for n in TRAIN_SIZES]
fig.add_traces(frames[0])
animate(fig, frames, STAGE_LABELS, prefix="CFD runs: ")
fig.update_layout(height=Sizes.HEIGHT + 110)
show(fig, "Epistemic shrinks with data; aleatoric does not", "left: what more runs remove · middle: what they cannot · right: the target")

<div style="border-left:5px solid #e9a23b;background:#fff7e8;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>💬 Discuss</b><br>You have 20 more CFD runs for this wing. Where do they go — for a <i>global surrogate</i>, and for <i>finding the best cruise condition</i>? Runs where the aleatoric map is bright mostly buy replicates.</div>

## 5. Bayesian optimization: uncertainty becomes a decision

An acquisition function scores every candidate from $(\mu, \sigma)$ alone; the next CFD run goes to its argmax.



Maximizing $L/D$, with $f^+$ the best so far and $z = (\mu - f^+)/\sigma$:

| rule | formula | character |
|---|---|---|
| PI | $\Phi(z)$ | greedy |
| **EI** | $\sigma\,[\,z\,\Phi(z) + \phi(z)\,]$ | weighs *how much* improvement |
| UCB | $\mu + \kappa\,\sigma$ | $\kappa$ dials exploration |

Every rule multiplies or adds σ: **no uncertainty, no exploration.** The σ that belongs here is the *epistemic* one — it rewards going where the model hasn't been, not where the CFD is noisy — with $f^+ = \max_i \mu(x_i)$ so a lucky noisy draw can't set the bar. The loop below starts from the 24-run Latin hypercube — exactly the situation the optimizer that produced the infill in `cfd_runs.csv` was in — and replays its first 16 decisions.

In [13]:
@dataclass(frozen=True)
class BOStep:
    """One iteration of the optimizer.

    Attributes:
        mach: Mach numbers run so far.
        aoa: Angles of attack run so far.
        mean: Surrogate mean on the grid.
        acquisition: Acquisition scores on the grid.
        next_index: Grid index of the next run.
        best_index: Grid index of the recommended condition (highest μ).
    """

    mach: FloatArray
    aoa: FloatArray
    mean: FloatArray
    acquisition: FloatArray
    next_index: int
    best_index: int


def acquisition(rule: Acquisition, mean: FloatArray, std: FloatArray, incumbent: float, kappa: float) -> FloatArray:
    """Score candidates under one rule (maximization).

    Args:
        rule: Acquisition rule.
        mean: Predictive mean.
        std: Predictive epistemic std.
        incumbent: Best value so far.
        kappa: Exploration weight (UCB only).

    Returns:
        Score per candidate.
    """
    z = (mean - incumbent) / np.maximum(std, 1e-9)
    if rule == Acquisition.PI:
        return norm.cdf(z)
    if rule == Acquisition.UCB:
        return mean + kappa * std
    return std * (z * norm.cdf(z) + norm.pdf(z))


def run_bo(rule: Acquisition = Acquisition.EI, kappa: float = 2.0, seed: int = Sizes.BO_SEED) -> list[BOStep]:
    """Surrogate-driven search for the best cruise condition from the 24-run Latin hypercube.

    Args:
        rule: Acquisition rule.
        kappa: Exploration weight (UCB only).
        seed: Observation-noise seed.

    Returns:
        One BOStep per iteration.
    """
    rng = np.random.default_rng(seed)
    mach, aoa, y = MACH_POOL[:Sizes.BO_INIT], AOA_POOL[:Sizes.BO_INIT], Y_POOL[:Sizes.BO_INIT]
    steps = []
    for _ in range(Sizes.BO_ITER):
        pipe = build_pipeline(seed=seed).fit(doe_table(mach, aoa), y)
        mean, std_epi, _ = predict_parts(pipe, X_GRID)
        acq = acquisition(rule, mean, std_epi, float(pipe.predict(doe_table(mach, aoa)).max()), kappa)
        nxt = int(np.argmax(acq))
        steps.append(BOStep(mach, aoa, mean, acq, nxt, int(np.argmax(mean))))
        mach, aoa = np.append(mach, MACH_GRID[nxt]), np.append(aoa, AOA_GRID[nxt])
        y = np.append(y, cfd_run(np.array([nxt]), rng))
    return steps


def regret(steps: list[BOStep]) -> FloatArray:
    """True L/D at the optimum minus true L/D at the recommended condition, per iteration.

    Args:
        steps: BO iterations.

    Returns:
        Regret per iteration.
    """
    return np.array([Y_TRUE[I_OPT] - Y_TRUE[s.best_index] for s in steps])


BO_STEPS = run_bo()
fig = fields_figure(("surrogate mean μ and CFD runs", "expected improvement"))
star = dict(name="next CFD run", color="#e76f51", size=16, symbol="star")
frames = [[field(fig, s.mean, 1, "Viridis", LD_RANGE, "μ"), points(s.mach[:Sizes.BO_INIT], s.aoa[:Sizes.BO_INIT], 1, name="Latin hypercube", symbol="diamond", size=9),
           points(s.mach[Sizes.BO_INIT:], s.aoa[Sizes.BO_INIT:], 1, name="runs added by BO", color="#1f5fa8", size=8), points(MACH_GRID[[s.next_index]], AOA_GRID[[s.next_index]], 1, **star),
           field(fig, s.acquisition, 2, "Teal", name="EI"), points(MACH_GRID[[s.next_index]], AOA_GRID[[s.next_index]], 2, **star)] for s in BO_STEPS]
fig.add_traces(frames[0])
animate(fig, frames, [str(i) for i in range(Sizes.BO_ITER)], prefix="BO iteration: ")
fig.add_trace(points(MACH_GRID[[I_OPT]], AOA_GRID[[I_OPT]], 1, name="true optimum", color="#1b1b1b", size=13, symbol="x"))
fig.update_layout(height=Sizes.HEIGHT + 110)
show(fig, "EI picks the next CFD run from μ and σ", f"it finds the ridge within a few runs — regret {regret(BO_STEPS)[-1]:.2f} L/D after {Sizes.BO_ITER} runs")

Same 24-run start and budget, EI versus random sampling, eight noise seeds. **Regret** = true $L/D$ at the optimum minus true $L/D$ at the recommended condition. Random gets lucky now and then; it cannot aim.

In [14]:
def regret_random(seed: int) -> FloatArray:
    """Regret per iteration when the extra runs are placed uniformly at random.

    Args:
        seed: Random seed for the choice of extra runs and their noise.

    Returns:
        Regret per iteration.
    """
    rng = np.random.default_rng(1000 + seed)
    picks = rng.choice(MACH_GRID.size, size=Sizes.BO_ITER, replace=False)
    mach = np.append(MACH_POOL[:Sizes.BO_INIT], MACH_GRID[picks])
    aoa = np.append(AOA_POOL[:Sizes.BO_INIT], AOA_GRID[picks])
    y = np.append(Y_POOL[:Sizes.BO_INIT], cfd_run(picks, rng))
    fits = [build_pipeline(seed=seed).fit(doe_table(mach[:k], aoa[:k]), y[:k]) for k in range(Sizes.BO_INIT, mach.size)]
    return np.array([Y_TRUE[I_OPT] - Y_TRUE[int(np.argmax(pipe.predict(X_GRID)))] for pipe in fits])


SEEDS = range(Sizes.BO_SEED, Sizes.BO_SEED + 8)
fig = go.Figure()
iters = np.arange(Sizes.BO_ITER)
for name, curves, color, fill in (("EI-driven BO", [regret(run_bo(seed=s)) for s in SEEDS], "#1f5fa8", "rgba(31,95,168,0.3)"),
                                   ("random sampling", [regret_random(s) for s in SEEDS], "#8a8a8a", "rgba(138,138,138,0.25)")):
    lo, med, hi = np.percentile(np.stack(curves), [25, 50, 75], axis=0)
    fig.add_scatter(x=np.concatenate([iters, iters[::-1]]), y=np.concatenate([hi, lo[::-1]]), fill="toself", fillcolor=fill, line_width=0, hoverinfo="skip", showlegend=False)
    fig.add_scatter(x=iters, y=med, line=dict(color=color, width=2.5), name=name)
fig.update_layout(height=360, xaxis_title="iteration", yaxis_title="regret [L/D]", hovermode="x unified", legend=dict(orientation="h", y=1.02, x=1, xanchor="right"))
show(fig, "Regret vs. iteration", "median and inter-quartile range over seeds")

<div style="border-left:5px solid #2a9d8f;background:#eef8f6;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>🔧 Your turn 4 — change the rule (5 min)</b><br><ol style='margin:6px 0 0 18px'><li>Run <code>run_bo(Acquisition.PI)</code> and UCB with <code>kappa=0.5</code> and <code>kappa=4</code>; print each final <code>regret</code>.</li><li>Print the last four runs of each. Who locks onto one condition and re-runs it? Who keeps probing?</li><li><i>Discuss:</i> what changes if the acquisition saw the <i>aleatoric</i> map from section 4 instead?</li></ol></div>

In [15]:
def last_runs(steps: list[BOStep], k: int = 4) -> list[tuple[float, float]]:
    """Mach/AoA of the last k CFD runs chosen.

    Args:
        steps: BO iterations.
        k: Number of final runs to report.

    Returns:
        List of (Mach, AoA) pairs.
    """
    return [(round(float(MACH_GRID[s.next_index]), 2), round(float(AOA_GRID[s.next_index]), 1)) for s in steps[-k:]]


print("EI  regret", f"{regret(BO_STEPS)[-1]:.2f}", last_runs(BO_STEPS))
# TODO: same for PI, UCB kappa=0.5, UCB kappa=4

EI  regret 0.08 [(0.76, 1.4), (0.2, 16.0), (0.38, 11.7), (0.75, 1.4)]


<details style="margin:10px 0"><summary style="cursor:pointer;color:#1f5fa8;font-weight:600">Solution</summary><pre style="background:#f5f7f9;border:1px solid #dde3ea;border-radius:6px;padding:12px;font-size:12.5px"><code>for label, steps in ((&quot;PI &quot;, run_bo(Acquisition.PI)), (&quot;UCB 0.5&quot;, run_bo(Acquisition.UCB, kappa=0.5)), (&quot;UCB 4  &quot;, run_bo(Acquisition.UCB, kappa=4.0))):
    print(label, f&quot;regret {regret(steps)[-1]:.2f}&quot;, last_runs(steps))</code></pre><p style='margin:8px 0 0 0'>All three rules find the ridge; they differ in the endgame — PI and greedy UCB lock onto one condition and re-run it, κ = 4 keeps probing, EI sits between. Re-running a known point is the sign to stop — that is what a confidence-bound stopping rule detects. With the aleatoric map in the loop, stall and buffet would look permanently interesting and the budget would go to replicates.</p></details>

## 6. Wrap-up

| here | in a production surrogate workflow |
|---|---|
| hand-built `StandardScaler → PCA → GP` | pipeline assembled and tuned by a hyperparameter search |
| `predict(X, return_std=True)` | the two outputs every surrogate should expose |
| exercise 3's broken coverage | coverage / CRPS in model selection; conformal recalibration |
| two-stage noise map | heteroscedastic GPs, BNN ensembles, conformalised tree ensembles |
| `run_bo` with EI on epistemic σ | Bayesian optimization with a confidence-bound stopping rule |

1. **μ is a guess; σ is how much to trust it** — report both, and check σ is honest.
2. **Epistemic uncertainty is a to-do list; aleatoric uncertainty is a fact of life.**
3. **Optimization without uncertainty is hill-climbing.**
<div style="border-left:5px solid #e9a23b;background:#fff7e8;padding:12px 16px;border-radius:6px;margin:14px 0;color:#222;line-height:1.5"><b>💬 Closing</b><br>Pick a case your team runs. Which DOE columns would a reducer collapse? Where does its noise come from — and is any of it solver noise you could reduce?</div>